# N05 — Interventional fidelity (개입적 충실도)

**Purpose**: For each (rejected customer, adverse feature) pair, check whether moving
to the adjacent less-adverse bin actually lowers P(default) in the teacher. Uses
`decentra.metrics.interventional.interventional_fidelity` on the scorecard derived from
Tree-d1.

**Outputs** (`outputs/N05/`): `interventional_results.json`

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pickle, json, numpy as np, pandas as pd
from pathlib import Path
import lightgbm as lgb
from decentra.surrogate import TreeSurrogate
from decentra.metrics.interventional import extract_bin_structure, interventional_fidelity

N01 = Path('../outputs/N01'); N02 = Path('../outputs/N02')
OUT = Path('../outputs/N05'); OUT.mkdir(parents=True, exist_ok=True)
with open(N01/'datasets.pkl','rb') as f: datasets = pickle.load(f)

# simple teacher wrapper: predict_proba from saved booster
class BoosterClf:
    def __init__(self, path): self.b = lgb.Booster(model_file=str(path))
    def predict_proba(self, X):
        p = self.b.predict(np.asarray(X))
        return np.column_stack([1-p, p])

rows = []
for name, d in datasets.items():
    score_tr = np.load(N02/f'bb_score_train_{name}.npy')
    prob_te = np.load(N02/f'bb_prob_{name}.npy')
    with open(N02/f'train_val_idx_{name}.pkl','rb') as f: tv = pickle.load(f)
    reject = prob_te >= np.percentile(prob_te, 90)

    surr = TreeSurrogate(max_depth=1, verbose=0)
    X_tr_i = d['X_train'].iloc[tv['train_pos']]
    X_val  = d['X_train'].iloc[tv['val_pos']]
    y_tr   = score_tr[tv['train_pos']]; y_val = score_tr[tv['val_pos']]
    surr.fit(X_tr_i, y_tr, eval_set=(X_val, y_val))

    bin_struct = extract_bin_structure(surr.model_, d['X_train'], len(d['feature_names']))
    teacher = BoosterClf(N02/f'bb_model_{name}.txt')
    contribs = np.asarray(surr.contributions(d['X_test']))
    out = interventional_fidelity(bin_struct, teacher, d['X_test'], reject, contribs)
    out['dataset'] = name
    rows.append(out)
    print(f'{name}: DA={out["DA"]:.3f}  rho={out["Spearman_rho"]:.3f}  '
          f'IR={out["IR"]:.3f}  n_pairs={out["n_pairs"]}')

with open(OUT/'interventional_results.json','w') as f:
    json.dump(rows, f, indent=2, default=float)